In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('F:\Soorykant\deidentification\deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [17]:
import csv
import json

# Define input CSV file and output JSON file
input_csv = "F:\\Soorykant\\deidentification\\NOTEBOOK\\phi_sheet.csv"

# Dictionary to store the result
table_dict = {}

# Read CSV file and process data
with open(input_csv, mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    headers = next(reader)  # Skip header row
    
    for row in reader:
        table_name, column_name = row[0], row[1]
        
        if table_name not in table_dict:
            table_dict[table_name] = []
        
        table_dict[table_name].append(column_name)


from nd_api.views.db_views import _get_default_table_details_for_ui

db_details_obj = DbDetailsModel.objects.get(db_name="prod_data")
for table, columns in table_dict.items():
    table_details_obj = TableDetailsModel.register_table(table, db_details_obj)
    table_details_obj.table_details_for_ui = _get_default_table_details_for_ui(
        columns
    )
    table_details_obj.save()

In [39]:
csv_file = "F:\\Soorykant\\MySQL Dump\\all_tables_rows.csv"  # Replace with your actual CSV filename

data_dict = {}

# Read CSV and convert to dictionary
with open(csv_file, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)  # Skip header row
    for row in reader:
        try:
            table_name, row_count = row[0], row[1]
            table_details_obj = TableDetailsModel.objects.get(table_name=table_name)
            table_details_obj.rows_count = row_count
            table_details_obj.save()
        except:
            pass
        


In [25]:
TableDetailsModel.objects.all()[520].rows_count

1910670

In [ ]:
TableDetailsModel

In [50]:
TableDetailsModel.objects.filter(rows_count__lte=10000).order_by('rows_count').count()

475

In [46]:
import csv
import json

# Define input CSV file and output JSON file
input_csv = "F:\\Soorykant\\deidentification\\NOTEBOOK\\phi_sheet.csv"

# Dictionary to store the result
required_tables = set()

# Read CSV file and process data
with open(input_csv, mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    headers = next(reader)  # Skip header row
    
    for row in reader:
        table_name, column_name = row[0], row[1]
        required_tables.add(table_name)

required_tables = list(required_tables)

In [58]:
required_tables[0]

'opprocedures'

In [63]:
c = 0
for t in TableDetailsModel.objects.filter(rows_count__lte=100000000).order_by('rows_count'):
    if t.table_name  in required_tables:
        c += 1
print(c)

476


In [67]:
for tb in TableDetailsModel.objects.all():
    json_dict = tb.table_details_for_ui
    
    json_dict['columns_details'].append(
        {'is_phi': False,
      'mask_value': '',
      'column_name': 'nd_auto_increament_id',
      'ignore_rows': {},
      'add_to_phi_table': False,
      'reference_mapping': {},
      'de_identification_rule': '',
      'column_name_for_phi_table': None}
    )
    tb.table_details_for_ui = json_dict
    tb.save()

In [77]:
TableDetailsModel.objects.filter(table_status=0).count()

385